# Zarr at scale: reads and writes from 10 MB to 10 GB

> **Docs:** [Zarr at scale notebook walkthrough](https://databrickslabs.github.io/geobrix/docs/notebooks/zarr-size-ladder) ·
> [Zarr Performance and Limits](https://databrickslabs.github.io/geobrix/docs/api/zarr-performance) ·
> [Zarr Reader](https://databrickslabs.github.io/geobrix/docs/readers/zarr) ·
> [Zarr Writer](https://databrickslabs.github.io/geobrix/docs/writers/zarr)

Weather models read Zarr and write Zarr. A wind model for one county reads a small grid for a long time.
A global model reads a grid of 11 km cells. This notebook shows what both cost as the store grows from 10 MB to 10 GB.

It makes one store, with the same variables as the [weather features notebook](https://databrickslabs.github.io/geobrix/docs/notebooks/zarr-weather-features)
(`t2m`, `u10`, `v10` and `msl`), in one of two shapes:

- **`county_wind`:** a 0.01 degree grid (about 1 km) over a one degree box, 101 by 101 cells, hourly.
- **`global_11km`:** the 0.1 degree global grid, 1,801 by 3,600 cells, hourly.

The `RUNG` setting picks the size, from 10 MB to 10 GB. The store grows along time.

1. **Read** the store five ways, and time each: the full table, a box over all time, 100 sites, wind speed with an hourly area mean, and one hour as a tile.
2. **Write a table to Zarr.** Read the cells, add `wind_speed` in Spark columns, and write them as a plain store and as a managed store.
3. **Append one more day** to both stores, then read both back and check that their sums equal the source.

The notebook ends with the timings of this run and the reference timings from DBR 19 and Serverless.
The outputs below are from a run on Serverless environment 6 with the 100 MB county store.

## Install GeoBrix

Zarr needs no JAR. The notebook installs the env6 wheel with the `zarr` extra.
It targets **Serverless environment 6**. On a classic **Databricks Runtime 19** cluster, install the same wheel and extra.

In [0]:
%pip install --quiet --disable-pip-version-check --no-deps --force-reinstall "geobrix @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.6.0+env6-py3-none-any.whl"
%pip install --quiet "geobrix[zarr] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.6.0+env6-py3-none-any.whl"

In [0]:
%restart_python

## Settings

Point `CATALOG`, `SCHEMA` and `WORK_DIR` at a schema and a Volume you can write to.

- `FAMILY` is `county_wind` or `global_11km`.
- `RUNG` is the size of the store: `10MB`, `50MB`, `100MB`, `500MB`, `1GB`, `2GB`, `5GB` or `10GB`.
- `GENERATE = True` makes the store when it is not in `WORK_DIR`, and reuses it when it is. `False` expects a store that is already there.
- `CLEAN_UP = True` drops this run's table and deletes its output folder at the end. `DROP_STORE = True` deletes the input store too.

In [0]:
CATALOG = "geospatial_docs"
SCHEMA = "geobrix"
WORK_DIR = "/Volumes/geospatial_docs/geobrix/sample-data/_examples/zarr-size-ladder"

FAMILY = "county_wind"  # "county_wind" or "global_11km"
RUNG = "100MB"  # 10MB, 50MB, 100MB, 500MB, 1GB, 2GB, 5GB or 10GB
GENERATE = True  # make the input store if it is missing
CLEAN_UP = True  # drop this run's table and delete its folder at the end
DROP_STORE = False  # also delete the input store

Every run gets its own `RUN_ID`, so its table and folder never collide with another run's.
The input store is in `WORK_DIR` under its family and rung. Each run writes its output under `WORK_DIR/runs`.

In [0]:
import uuid

RUN_ID = uuid.uuid4().hex[:8]
STORE_PATH = f"{WORK_DIR}/{FAMILY}/{RUNG}.zarr"  # the input store
RUN_DIR = f"{WORK_DIR}/runs/{RUN_ID}"  # this run's output
PLAIN = f"{RUN_DIR}/plain.zarr"  # the plain output store
MANAGED = f"{CATALOG}.{SCHEMA}.zarr_ladder_{RUN_ID}"  # the managed output store's manifest table
print("input store   :", STORE_PATH)
print("plain store   :", PLAIN)
print("managed store :", MANAGED)

input store   : /Volumes/geospatial_docs/geobrix/sample-data/_examples/zarr-size-ladder/county_wind/100MB.zarr
plain store   : /Volumes/geospatial_docs/geobrix/sample-data/_examples/zarr-size-ladder/runs/fbbfce1c/plain.zarr
managed store : geospatial_docs.geobrix.zarr_ladder_fbbfce1c


## The shape of the store

A rung is the uncompressed size of the four variables in decimal MB or GB. The store grows along time, one hour at a time.

- **`county_wind`:** one hour of the box is 0.16 MB, so 100 MB is about 600 hours. Chunks hold 24 hours of the whole box. Shards hold 168 hours.
- **`global_11km`:** one hour of the globe is 104 MB. A rung below 500 MB is one hour of a window of the grid, in the grid's 1:2 shape. From 500 MB it is whole-globe hours. Chunks are 24 hours by 90 by 90 cells. Shards are 168 by 360 by 360.

`ladder_shape` turns the family and the rung into the number of hours and the window.

In [0]:
import math

import numpy as np

VARIABLES = ("t2m", "u10", "v10", "msl")
DIMS = ("time", "latitude", "longitude")
# step: grid step in degrees; box: (north, south, west, east) of the store, None for the globe;
# area: the box the wind-model reads cut; sites: where the 100 sites are drawn
FAMILIES = {
    "county_wind": {
        "step": 0.01,
        "box": (40.0, 39.0, 255.0, 256.0),
        "chunks": (24, 101, 101),
        "shards": (168, 101, 101),
        "area": (39.8, 39.2, 255.2, 255.8),
        "sites": (39.99, 39.01, 255.01, 255.99),
    },
    "global_11km": {
        "step": 0.1,
        "box": None,
        "chunks": (24, 90, 90),
        "shards": (168, 360, 360),
        "area": (2.5, -2.5, 177.5, 182.5),
        "sites": (4.9, -4.9, 175.1, 184.9),
    },
}
FAM = FAMILIES[FAMILY]


def axis_labels(family):
    """Latitudes (from the north) and longitudes (from 0 east) of the family's store."""
    f = FAMILIES[family]
    lat = np.linspace(90.0, -90.0, round(180 / f["step"]) + 1)
    lon = np.arange(round(360 / f["step"])) * f["step"]
    if f["box"]:
        north, south, west, east = f["box"]
        tol = f["step"] * 1e-6
        lat = lat[(lat <= north + tol) & (lat >= south - tol)]
        lon = lon[(lon >= west - tol) & (lon <= east + tol)]
    return lat, lon


def ladder_shape(family, rung):
    """The hours of the rung and the window (north, south, west, east) of the grid it covers."""
    f = FAMILIES[family]
    target = float(rung[:-2]) * {"MB": 1e6, "GB": 1e9}[rung[-2:]]
    lat, lon = axis_labels(family)
    ny, nx = lat.size, lon.size
    cell_bytes = 4 * len(VARIABLES)  # float32
    if target >= ny * nx * cell_bytes:
        return max(1, round(target / (ny * nx * cell_bytes))), f["box"]
    # below one hour of the whole grid: one hour of a centred window with the grid's shape
    cells = target / cell_bytes
    wy = max(1, min(ny, round(math.sqrt(cells * ny / nx))))
    wx = max(1, min(nx, round(cells / wy)))
    r0, c0 = (ny - wy) // 2, (nx - wx) // 2
    window = (float(lat[r0]), float(lat[r0 + wy - 1]), float(lon[c0]), float(lon[c0 + wx - 1]))
    return 1, window


HOURS_WANTED, WINDOW = ladder_shape(FAMILY, RUNG)
print(f"{FAMILY} {RUNG}: {HOURS_WANTED} hours, window {WINDOW}")

county_wind 100MB: 613 hours, window (40.0, 39.0, 255.0, 256.0)


## Make or reuse the store

`write_era5_cube` writes the store the way a data vendor does: Zarr v3, sharded, zstd, float32, with CF times.
It builds the store one shard at a time, so a 10 GB store does not need 10 GB of memory.
`noise` and `precision` make the values compress like real weather data, about two to three fold with zstd.
Without them the made-up fields would compress about 50 fold and every read would look cheaper than it is.

The notebook then reads the store's labels with xarray. They name every cell, and the writes below reuse them.

In [0]:
import pathlib
import time

import xarray as xr

from databricks.labs.gbx.sample.zarr_era5 import write_era5_cube

if pathlib.Path(STORE_PATH, "zarr.json").exists():
    print("reusing", STORE_PATH)
elif not GENERATE:
    raise FileNotFoundError(f"{STORE_PATH} is missing. Fix: set GENERATE = True, or point WORK_DIR at your store.")
else:
    started = time.perf_counter()
    write_era5_cube(
        f"{WORK_DIR}/{FAMILY}",
        hours=HOURS_WANTED,
        step=FAM["step"],
        box=WINDOW,
        chunks=FAM["chunks"],
        shards=FAM["shards"],
        v3_name=f"{RUNG}.zarr",
        noise=1.0,
        precision=0.05,
    )
    print(f"made {STORE_PATH} in {time.perf_counter() - started:.1f} s")

ds = xr.open_zarr(STORE_PATH, consolidated=False, chunks=None)
LABELS = {d: ds[d].values for d in DIMS}
HOURS, NY, NX = (len(LABELS[d]) for d in DIMS)
STORE_BYTES = sum(p.stat().st_size for p in pathlib.Path(STORE_PATH).rglob("*") if p.is_file())
print(f"store: {HOURS} hours x {NY} x {NX} cells, {HOURS * NY * NX:,} cells per variable")
print(f"logical size {HOURS * NY * NX * 4 * len(VARIABLES) / 1e6:,.0f} MB, on disk {STORE_BYTES / 1e6:,.0f} MB")

made /Volumes/geospatial_docs/geobrix/sample-data/_examples/zarr-size-ladder/county_wind/100MB.zarr in 18.3 s
store: 613 hours x 101 x 101 cells, 6,253,213 cells per variable
logical size 100 MB, on disk 40 MB


## 1. Reads

Each read below prints one line: the step, its wall-clock time and what it returned.
A one-hour warm-up read comes first. It starts the Zarr readers, so the times below leave out their start-up.

- **Full table:** `gbx_zarr_read(output="table")` returns one row per cell, with a column per dimension and per variable. The count forces the whole read.
- **A box over all time:** `sel` selects by label. This is the input of a wind model for one area.
- **100 sites:** `gbx_zarr_sample` reads the nearest cell at each point, for every hour.
- **Wind speed:** `sqrt(u10² + v10²)` in Spark columns, then the mean of the area for each hour.
- **One hour as a tile:** `output="tiles"` returns a GeoBrix raster tile of the whole grid for one hour.

In [0]:
from pyspark.sql import functions as F

from databricks.labs.gbx.zarr import gbx_zarr_read, gbx_zarr_sample

RESULTS = []  # (step, seconds)


def timed(step, fn):
    """Run fn, print one line with its wall-clock time and what it returned."""
    started = time.perf_counter()
    detail = fn()
    seconds = time.perf_counter() - started
    RESULTS.append((step, seconds))
    print(f"{step:<34}{seconds:8.2f} s   {detail}")


def n_between(values, high, low):
    """How many labels lie in [low, high], with a small tolerance for float labels."""
    tol = abs(high - low) * 1e-9 + 1e-9
    return int(((values <= high + tol) & (values >= low - tol)).sum())


north, south, west, east = FAM["area"]
AREA = {"latitude": slice(north, south), "longitude": slice(west, east)}
N_AREA = n_between(LABELS["latitude"], north, south) * n_between(LABELS["longitude"], east, west)
assert N_AREA > 0, "the area holds no cell of this store"


def full_table():
    n = gbx_zarr_read(spark, STORE_PATH, output="table").count()
    assert n == HOURS * NY * NX, n
    return f"{n:,} cells"


def box_over_time():
    n = gbx_zarr_read(spark, STORE_PATH, sel=AREA, output="table").count()
    assert n == HOURS * N_AREA, n
    return f"{n:,} cells"


def hundred_sites():
    n_lat, s_lat, w_lon, e_lon = FAM["sites"]
    rng = np.random.default_rng(7)
    lat = rng.uniform(s_lat, n_lat, 100).round(2)
    lon = rng.uniform(w_lon, e_lon, 100).round(2)
    sites = spark.createDataFrame(
        [(f"site-{i:03d}", f"POINT({x} {y})") for i, (x, y) in enumerate(zip(lon, lat))],
        ["site", "geom"],
    )
    n = gbx_zarr_sample(spark, sites, STORE_PATH, variables=list(VARIABLES)).count()
    assert n == 100 * HOURS, n
    return f"{n:,} rows (100 sites x {HOURS} hours)"


def wind_speed_hourly_mean():
    u, v = F.col("u10").cast("double"), F.col("v10").cast("double")
    hourly = (
        gbx_zarr_read(spark, STORE_PATH, variables=["u10", "v10"], sel=AREA, output="table")
        .groupBy("time")
        .agg(F.avg(F.sqrt(u * u + v * v)).alias("mean_speed"))
        .orderBy("time")
        .collect()
    )
    assert len(hourly) == HOURS, len(hourly)
    return f"{len(hourly)} hourly means, the first {hourly[0]['mean_speed']:.2f} m/s"


def one_hour_tile():
    hour = str(LABELS["time"][HOURS // 2])[:16]
    tiles = gbx_zarr_read(
        spark,
        STORE_PATH,
        variables=["u10"],
        sel={"time": hour},
        output="tiles",
        tile_size=(NX, NY),
        virtual_tiles=False,
    )
    n = tiles.count()
    assert n == 1, n
    return f"{n} tile of {NX} x {NY} cells at {hour}"


# warm-up: a one-hour read starts the Zarr readers on the cluster, outside the timings
gbx_zarr_read(spark, STORE_PATH, variables=["u10"], isel={"time": slice(0, 1)}, output="table").count()

timed("Read the full table", full_table)
timed("Read a box over all time", box_over_time)
timed("Sample 100 sites", hundred_sites)
timed("Wind speed, hourly area mean", wind_speed_hourly_mean)
timed("Read one hour as a tile", one_hour_tile)

Read the full table                  14.27 s   6,253,213 cells
Read a box over all time              4.25 s   2,280,973 cells
Sample 100 sites                      5.68 s   61,300 rows (100 sites x 613 hours)
Wind speed, hourly area mean          5.12 s   613 hourly means, the first 1.68 m/s
Read one hour as a tile               3.77 s   1 tile of 101 x 101 cells at 2023-01-13T18:00


## 2. Table to Zarr

A weather model reads cells, computes new columns and writes the result as a new Zarr store.
The steps are:

1. Read the store as a table of cells, `gbx_zarr_read(output="table")`.
2. Add `wind_speed` in Spark columns.
3. Write the table with `gbx_zarr_write`. `dim_cols` names the column that holds each cell's label along a dimension.

Two arguments let the writer plan without a scan of the table. `coords` gives the labels of every dimension, and `complete=True` says every storage unit has cells.
The writer then reads the table once, in the write itself.
`chunks` and `shards` give the layout of the source store.

**Plain or managed.** `transactional=False` writes a *plain* store at a path. It has no manifest table, no versions and no commit, and other tools read it as it is.
Use it for one writer at a time, when speed matters, or when the store is for other tools.
A *managed* store, the default, adds a manifest table. It gives you all-or-nothing writes, versions, time travel and safe concurrent writers.
Use it when more than one job writes, or when you need to read an earlier version.

In [0]:
import os

from databricks.labs.gbx.zarr import gbx_zarr_write

OUT_VARS = ["t2m", "u10", "v10", "wind_speed"]
DIM_COLS = {d: d for d in DIMS}
CHUNKS = dict(zip(DIMS, FAM["chunks"]))
SHARDS = dict(zip(DIMS, FAM["shards"]))
os.makedirs(RUN_DIR, exist_ok=True)


def cells_of(hours=None, shift_hours=0):
    """The store's cells with wind_speed added. `hours` keeps the first hours; `shift_hours` moves the times on."""
    kwargs = {} if hours is None else {"isel": {"time": slice(0, hours)}}
    cells = gbx_zarr_read(spark, STORE_PATH, variables=["t2m", "u10", "v10"], output="table", **kwargs)
    u, v = F.col("u10").cast("double"), F.col("v10").cast("double")
    cells = cells.withColumn("wind_speed", F.sqrt(u * u + v * v).cast("float"))
    if shift_hours:
        cells = cells.withColumn("time", F.col("time") + F.expr(f"INTERVAL {shift_hours} HOURS"))
    return cells.select(*DIMS, *OUT_VARS)


def totals(cells):
    """The number of cells and the sum of every variable."""
    sums = [F.sum(F.col(c).cast("double")).alias(c) for c in OUT_VARS]
    return cells.agg(F.count(F.lit(1)).alias("n"), *sums).first().asDict()


def same(got, want):
    return got["n"] == want["n"] and all(
        abs(got[c] - want[c]) <= 1e-9 * max(1.0, abs(want[c])) for c in OUT_VARS
    )


def size_mb(path):
    return sum(p.stat().st_size for p in pathlib.Path(path).rglob("*") if p.is_file()) / 1e6


EXPECTED = totals(cells_of())
print(f"source cells: {EXPECTED['n']:,}, sum of wind_speed {EXPECTED['wind_speed']:.6g}")


def write_plain():
    gbx_zarr_write(
        cells_of(),
        PLAIN,
        dim_cols=DIM_COLS,
        coords=LABELS,
        complete=True,
        chunks=CHUNKS,
        shards=SHARDS,
        transactional=False,
        spark=spark,
    )
    return f"{size_mb(PLAIN):,.0f} MB on disk"


def write_managed():
    version = gbx_zarr_write(
        cells_of(),
        MANAGED,
        mode="create",
        root=f"{RUN_DIR}/managed",
        dim_cols=DIM_COLS,
        coords=LABELS,
        complete=True,
        chunks=CHUNKS,
        shards=SHARDS,
        spark=spark,
    )
    return f"store version {version}, {size_mb(f'{RUN_DIR}/managed'):,.0f} MB on disk"


timed("Write a plain store", write_plain)
timed("Write a managed store", write_managed)

for name, target in (("plain", PLAIN), ("managed", MANAGED)):
    got = totals(gbx_zarr_read(spark, target, output="table"))
    assert same(got, EXPECTED), (name, got, EXPECTED)
    print(f"{name} store read back: {got['n']:,} cells, sums equal the source")

source cells: 6,253,213, sum of wind_speed 2.09693e+07
Write a plain store                  22.63 s   49 MB on disk
Write a managed store                35.66 s   store version 1, 49 MB on disk
plain store read back: 6,253,213 cells, sums equal the source
managed store read back: 6,253,213 cells, sums equal the source


## 3. Append one more day

A forecast store grows. `mode="append"` with `append_dim="time"` adds the new hours after the store's last hour.
Here the next day is the first 24 hours of the source with their times moved on by the length of the store.
`coords` holds only the new labels of the append dimension. The other dimensions use the store's labels.

A one-hour store has no day to append, so the smallest global rungs skip this step.
After the appends, both stores are read back and compared with the source plus the new day.

In [0]:
DAY = min(24, HOURS)
if HOURS < 2:
    print("skipped: this store has one time step, so there is no day to append")
else:
    DAY_LABELS = {"time": (LABELS["time"][:DAY] + np.timedelta64(HOURS, "h")).astype("datetime64[ns]")}
    next_day = cells_of(hours=DAY, shift_hours=HOURS)
    expected_after = {k: EXPECTED[k] + v for k, v in totals(next_day).items()}

    def append_plain():
        gbx_zarr_write(
            next_day,
            PLAIN,
            mode="append",
            append_dim="time",
            dim_cols=DIM_COLS,
            coords=DAY_LABELS,
            complete=True,
            transactional=False,
            spark=spark,
        )
        return f"{DAY} hours added"

    def append_managed():
        version = gbx_zarr_write(
            next_day,
            MANAGED,
            mode="append",
            append_dim="time",
            dim_cols=DIM_COLS,
            coords=DAY_LABELS,
            complete=True,
            spark=spark,
        )
        return f"{DAY} hours added, store version {version}"

    timed("Append a day, plain", append_plain)
    timed("Append a day, managed", append_managed)

    for name, target in (("plain", PLAIN), ("managed", MANAGED)):
        got = totals(gbx_zarr_read(spark, target, output="table"))
        assert same(got, expected_after), (name, got, expected_after)
        print(f"{name} store read back: {got['n']:,} cells, sums equal the source plus the new day")

Append a day, plain                   5.67 s   24 hours added
Append a day, managed                19.84 s   24 hours added, store version 2
plain store read back: 6,498,037 cells, sums equal the source plus the new day
managed store read back: 6,498,037 cells, sums equal the source plus the new day


## Timings of this run

All the steps, with the same labels as above. This is one run, and the smallest stores are dominated by fixed costs such as starting tasks.
The reference timings below are medians of three runs.

In [0]:
import pandas as pd

summary = pd.DataFrame(RESULTS, columns=["step", "seconds"]).round(2)
print(f"{FAMILY} {RUNG}: {HOURS} hours x {NY} x {NX} cells, {STORE_BYTES / 1e6:,.0f} MB on disk")
print(summary.to_string(index=False))

county_wind 100MB: 613 hours x 101 x 101 cells, 40 MB on disk
                        step  seconds
         Read the full table    14.27
    Read a box over all time     4.25
            Sample 100 sites     5.68
Wind speed, hourly area mean     5.12
     Read one hour as a tile     3.77
         Write a plain store    22.63
       Write a managed store    35.66
         Append a day, plain     5.67
       Append a day, managed    19.84


## Reference timings

These timings were measured on two kinds of compute. **Your numbers will differ.** Compute, the Volume, and the
other work on the cluster all change them.

- **DBR 19:** Databricks Runtime 19, two workers of four cores each (`m6gd.xlarge`), dedicated access.
- **Serverless:** Serverless environment 6. Its compute varies from run to run, so use these numbers to see the
  shape of the cost, not to compare with a fixed cluster.

Each time is the median of three runs, in seconds, for the steps in this notebook. The first run is cold. The
global stores below 500 MB are one hour of a regional window, so "100 MB" global is a different shape from "1 GB".

**County wind** (0.01 degree grid over a one degree box; 24 hours in a chunk)

| Store | Compute | Full table | Box | 100 sites | Wind speed | One-hour tile | Write plain | Write managed |
|---|---|---|---|---|---|---|---|---|
| 100 MB | DBR 19 | 2.9 s | 2.5 s | 2.7 s | 2.3 s | 2.7 s | 9.8 s | 23.6 s |
| 100 MB | Serverless | 3.2 s | 3.0 s | 3.3 s | 2.9 s | 2.2 s | 10.6 s | 16.0 s |
| 1 GB | DBR 19 | 13.1 s | 9.7 s | 7.1 s | 6.9 s | 3.1 s | 37.8 s | 40.6 s |
| 1 GB | Serverless | 14.9 s | 13.7 s | 10.0 s | 9.5 s | 2.3 s | 41.7 s | 45.8 s |
| 10 GB | DBR 19 | 113.3 s | 86.0 s | 53.0 s | 53.7 s | 3.3 s | 333.6 s | 307.8 s |
| 10 GB | Serverless | 55.0 s | 42.6 s | 18.7 s | 22.7 s | 3.2 s | 112.3 s | 149.6 s |

**Global 11 km** (0.1 degree grid)

| Store | Compute | Full table | Box | 100 sites | Wind speed | One-hour tile | Write plain | Write managed |
|---|---|---|---|---|---|---|---|---|
| 100 MB | DBR 19 | 12.8 s | 1.5 s | 1.2 s | 1.3 s | 4.6 s | 21.2 s | 21.4 s |
| 100 MB | Serverless | 18.3 s | 2.2 s | 1.7 s | 2.0 s | 5.3 s | 32.0 s | 38.3 s |
| 1 GB | DBR 19 | 23.8 s | 2.7 s | 1.4 s | 1.7 s | 4.4 s | 45.2 s | 48.5 s |
| 1 GB | Serverless | 13.2 s | 5.8 s | 2.0 s | 2.1 s | 6.8 s | 22.0 s | 30.6 s |
| 10 GB | DBR 19 | 151.3 s | 2.4 s | 3.0 s | 2.0 s | 10.0 s | 284.6 s | 297.4 s |
| 10 GB | Serverless | 54.6 s | 2.9 s | 2.9 s | 2.5 s | 7.8 s | 94.8 s | 71.9 s |

What the numbers show:

- **A full-table read grows with the size of the store.** At 10 GB it takes 113 s (county) and 151 s (global) on DBR 19, and about 55 s on Serverless.
- **Reads of a part of the store stay short.** On the global grid the box, the 100 sites, the wind speed and the one-hour tile take 10 s or less at every rung. They read only the chunks they need.
- **A county store has one spatial chunk.** Its box, site and wind-speed reads therefore grow with the length of the series.
- **Plain and managed writes are close from 1 GB.** At 100 MB the plain write is faster, by a fixed cost of seconds that the manifest and the commit add.

## Clean up

`CLEAN_UP = True` drops this run's managed table and deletes its output folder: the plain store and the managed store's files.
The input store stays in `WORK_DIR`, so the next run reuses it. Set `DROP_STORE = True` to delete it as well.

In [0]:
import shutil

if CLEAN_UP:
    spark.sql(f"DROP TABLE IF EXISTS {MANAGED}")
    shutil.rmtree(RUN_DIR, ignore_errors=True)
    if DROP_STORE:
        shutil.rmtree(STORE_PATH, ignore_errors=True)
    left = [MANAGED] if spark.catalog.tableExists(MANAGED) else []
    left += [p for p in (RUN_DIR, STORE_PATH if DROP_STORE else None) if p and pathlib.Path(p).exists()]
    print("not removed:", ", ".join(left) if left else "nothing")
else:
    print("kept", MANAGED, RUN_DIR)

not removed: nothing
